# Add Noise and Denoise Custom Images

Upload your own image, add noise, and test denoising techniques

## Setup

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from tkinter import Tk, filedialog
from PIL import Image

%matplotlib inline

print("Setup complete")

## Upload Image and Add Noise

In [ ]:
# Open file dialog to select image
root = Tk()
root.withdraw()
root.attributes('-topmost', True)

file_path = filedialog.askopenfilename(
    title='Select an image',
    filetypes=[('Image files', '*.jpg *.jpeg *.png *.bmp')]
)

if file_path:
    # Load image
    img = Image.open(file_path)
    img = img.convert('RGB')
    original = np.array(img)
    
    print(f"Loaded: {file_path}")
    print(f"Image shape: {original.shape}")
    
    # Add Gaussian noise
    noise_level = 25
    noise = np.random.normal(0, noise_level, original.shape)
    noisy_image = np.clip(original + noise, 0, 255).astype(np.uint8)
    
    # Apply denoising techniques
    print("Applying denoising techniques...")
    denoised_gaussian = cv2.GaussianBlur(noisy_image, (5, 5), 0)
    denoised_bilateral = cv2.bilateralFilter(noisy_image, 9, 75, 75)
    denoised_nlm = cv2.fastNlMeansDenoisingColored(noisy_image, None, 10, 10, 7, 21)
    
    # Calculate metrics
    psnr_noisy = peak_signal_noise_ratio(original, noisy_image)
    psnr_gaussian = peak_signal_noise_ratio(original, denoised_gaussian)
    psnr_bilateral = peak_signal_noise_ratio(original, denoised_bilateral)
    psnr_nlm = peak_signal_noise_ratio(original, denoised_nlm)
    
    ssim_noisy = structural_similarity(original, noisy_image, channel_axis=2)
    ssim_gaussian = structural_similarity(original, denoised_gaussian, channel_axis=2)
    ssim_bilateral = structural_similarity(original, denoised_bilateral, channel_axis=2)
    ssim_nlm = structural_similarity(original, denoised_nlm, channel_axis=2)
    
    # Display results
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    axes[0, 0].imshow(original)
    axes[0, 0].set_title('Original (Clean)', fontsize=12)
    axes[0, 0].axis('off')
    
    axes[0, 1].imshow(noisy_image)
    axes[0, 1].set_title(f'Noisy Image\nPSNR: {psnr_noisy:.2f} dB | SSIM: {ssim_noisy:.3f}', fontsize=12)
    axes[0, 1].axis('off')
    
    axes[0, 2].imshow(denoised_gaussian)
    axes[0, 2].set_title(f'Gaussian Denoised\nPSNR: {psnr_gaussian:.2f} dB | SSIM: {ssim_gaussian:.3f}', fontsize=12)
    axes[0, 2].axis('off')
    
    axes[1, 0].imshow(denoised_bilateral)
    axes[1, 0].set_title(f'Bilateral Denoised\nPSNR: {psnr_bilateral:.2f} dB | SSIM: {ssim_bilateral:.3f}', fontsize=12)
    axes[1, 0].axis('off')
    
    axes[1, 1].imshow(denoised_nlm)
    axes[1, 1].set_title(f'NLM Denoised\nPSNR: {psnr_nlm:.2f} dB | SSIM: {ssim_nlm:.3f}', fontsize=12)
    axes[1, 1].axis('off')
    
    noise_diff = np.abs(original.astype(float) - noisy_image.astype(float))
    axes[1, 2].imshow(noise_diff.astype(np.uint8))
    axes[1, 2].set_title('Noise Pattern', fontsize=12)
    axes[1, 2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\nDenoising Results:")
    print(f"Noisy Image - PSNR: {psnr_noisy:.2f} dB, SSIM: {ssim_noisy:.3f}")
    print(f"Gaussian    - PSNR: {psnr_gaussian:.2f} dB, SSIM: {ssim_gaussian:.3f} (improvement: {psnr_gaussian - psnr_noisy:.2f} dB)")
    print(f"Bilateral   - PSNR: {psnr_bilateral:.2f} dB, SSIM: {ssim_bilateral:.3f} (improvement: {psnr_bilateral - psnr_noisy:.2f} dB)")
    print(f"NLM         - PSNR: {psnr_nlm:.2f} dB, SSIM: {ssim_nlm:.3f} (improvement: {psnr_nlm - psnr_noisy:.2f} dB)")
    
else:
    print("No file selected")

## Adjust Noise Level and Reprocess

In [ ]:
# Change noise level and reprocess
noise_level = 50  # Adjust this value (0-100)

if 'original' in locals():
    # Add noise with new level
    noise = np.random.normal(0, noise_level, original.shape)
    noisy_image = np.clip(original + noise, 0, 255).astype(np.uint8)
    
    # Denoise
    denoised_nlm = cv2.fastNlMeansDenoisingColored(noisy_image, None, 10, 10, 7, 21)
    
    # Calculate PSNR
    psnr_noisy = peak_signal_noise_ratio(original, noisy_image)
    psnr_nlm = peak_signal_noise_ratio(original, denoised_nlm)
    
    # Display comparison
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    axes[0].imshow(original)
    axes[0].set_title('Original')
    axes[0].axis('off')
    
    axes[1].imshow(noisy_image)
    axes[1].set_title(f'Noisy (level={noise_level})\nPSNR: {psnr_noisy:.2f} dB')
    axes[1].axis('off')
    
    axes[2].imshow(denoised_nlm)
    axes[2].set_title(f'NLM Denoised\nPSNR: {psnr_nlm:.2f} dB')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"Noise level: {noise_level}")
    print(f"PSNR improvement: {psnr_nlm - psnr_noisy:.2f} dB")
else:
    print("Please run the previous cell first to load an image")